In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pandas as pd
import numpy as np

from src.config import PATHS
from src.viz import set_style, save_fig, annotate_source

set_style()

In [ ]:
df_h = pd.read_parquet(PATHS['analytical'] / 'df_hogar_analytic.parquet')
df_p = pd.read_parquet(PATHS['analytical'] / 'df_persona_analytic.parquet')

print(f'Hogares analíticos : {df_h.shape[0]:,} observaciones × {df_h.shape[1]} variables')
print(f'Personas analíticas: {df_p.shape[0]:,} observaciones × {df_p.shape[1]} variables')

## 8.2 Dificultad Económica $\times$ Brecha Digital en el Hogar

- Acceso general a Internet (`acceso_internet_hogar`).
- Tenencia de computador en el hogar (`tiene_computador`).

In [ ]:
# Tabla de contingencia: Dificultad Económica vs Acceso a Internet
ct_eco_net = pd.crosstab(df_h['sit_eco_label'], df_h['acceso_internet_hogar'], margins=True)
pct_eco_net = pd.crosstab(df_h['sit_eco_label'], df_h['acceso_internet_hogar'], normalize='index') * 100

print('Conteos observados (n_ij):')
print(ct_eco_net)
print('\nDistribución porcentual por fila P(Internet | Situación económica) (%):')
print(pct_eco_net.round(2))

# Tabla de contingencia: Dificultad Económica vs Computador
pct_eco_pc = pd.crosstab(df_h['sit_eco_label'], df_h['tiene_computador'], normalize='index') * 100
print('Distribución porcentual por fila P(Computador | Situación económica) (%):')
print(pct_eco_pc.round(2))

### Gráfico 6: Tenencia de Computador según Nivel de Dificultad Económica

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
pc_eco = df_h.groupby('sit_eco_label')['tiene_computador'].mean() * 100
colores_grad = ['#2ca02c', '#74c476', '#fdae6b', '#f16913', '#d94801']
bars = ax.bar(pc_eco.index, pc_eco.values, color=colores_grad, width=0.55, edgecolor='black', alpha=0.88)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 85)
ax.set_title('Tenencia de Computador en el Hogar según Nivel de Dificultad Económica', pad=15)
ax.set_ylabel('Hogares con computador (%)')
ax.set_xlabel('Facilidad / Dificultad reportada para cubrir gastos mínimos', labelpad=10)
for b in bars:
 h = b.get_height()
 ax.text(b.get_x() + b.get_width()/2, h + 1.0, f'{h:.1f}%', ha='center', fontweight='bold', fontsize=9.5)
plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — Encuesta de Calidad de Vida (ECV) 2024. Muestra: 85,387 hogares.')
save_fig(fig, 'fig06_bivariado_computador_por_situacion_economica')
plt.show()

## 8.3 Capital Humano del Jefe de Hogar $\times$ Conectividad y Dispositivos

In [ ]:
educ_order = ['1. Ninguno / Preescolar', '2. Primaria', '3. Secundaria', '4. Media', '5. Superior / Universitario']
df_h_valid_educ = df_h[df_h['jefe_nivel_educ'].isin(educ_order)]

tab_educ = df_h_valid_educ.groupby('jefe_nivel_educ').agg(
 n=('id_hogar', 'count'),
 pct_internet=('acceso_internet_hogar', lambda x: x.mean() * 100),
 pct_computador=('tiene_computador', lambda x: x.mean() * 100),
 pct_fijo=('tiene_internet_fijo', lambda x: x.mean() * 100)
).loc[educ_order]

print(tab_educ.round(2))

### Gráfico 7: Conectividad y Computador según Nivel Educativo del Jefe de Hogar

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(educ_order))
w = 0.35
b1 = ax.bar(x - w/2, tab_educ['pct_internet'], width=w, label='Acceso a Internet en el hogar', color='#1f77b4', edgecolor='white')
b2 = ax.bar(x + w/2, tab_educ['pct_computador'], width=w, label='Tenencia de computador', color='#aec7e8', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(['Ninguno', 'Primaria', 'Secundaria', 'Media', 'Superior'], fontsize=9.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 110)
ax.set_title('Conectividad y Computador según Nivel Educativo del Jefe de Hogar', pad=15)
ax.set_ylabel('Porcentaje de hogares (%)')
ax.set_xlabel('Nivel educativo del jefe del hogar', labelpad=10)
ax.legend(frameon=True, facecolor='white', loc='upper left')

for b in b1:
 h = b.get_height()
 ax.text(b.get_x() + b.get_width()/2, h + 1.2, f'{h:.1f}%', ha='center', fontsize=8.5, fontweight='bold', color='#1f77b4')
for b in b2:
 h = b.get_height()
 ax.text(b.get_x() + b.get_width()/2, h + 1.2, f'{h:.1f}%', ha='center', fontsize=8.5, fontweight='bold', color='#3182bd')

plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Muestra analizada: 83,559 hogares con jefe identificado.')
save_fig(fig, 'fig07_bivariado_educacion_jefe_vs_dispositivos')
plt.show()

## 8.4 Nivel Educativo Individual $\times$ Uso de Internet en Personas

In [ ]:
peduc_order = ['1. Ninguno / Preescolar', '2. Primaria', '3. Secundaria', '4. Media', '5. Superior / Universitario']
df_p_valid = df_p[df_p['nivel_educ_agrupado'].isin(peduc_order)]

ct_pedu_net = pd.crosstab(df_p_valid['nivel_educ_agrupado'], df_p_valid['usa_internet'], margins=True)
pct_pedu_net = pd.crosstab(df_p_valid['nivel_educ_agrupado'], df_p_valid['usa_internet'], normalize='index') * 100

print('Conteos observados:')
print(ct_pedu_net)
print('\nDistribución porcentual por fila P(Usa Internet | Educación) (%):')
print(pct_pedu_net.round(2))

### Gráfico 8: Tasa de Uso Individual de Internet según Nivel Educativo Alcanzado

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
uso_peduc = df_p_valid.groupby('nivel_educ_agrupado')['usa_internet'].mean().loc[peduc_order] * 100

bars = ax.barh(['Ninguno', 'Primaria', 'Secundaria', 'Media', 'Superior'], uso_peduc.values, color='#08519c', height=0.52)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlim(0, 110)
ax.set_title('Tasa de Uso Individual de Internet según Nivel Educativo Alcanzado', pad=15)
ax.set_xlabel('Personas que usan Internet (%)', labelpad=8)
ax.set_ylabel('Nivel educativo de la persona')
for b in bars:
 w = b.get_width()
 ax.text(w + 1.2, b.get_y() + b.get_height()/2, f'{w:.1f}%', va='center', fontweight='bold', fontsize=9.5)
plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Módulo TIC y Educación. Muestra: 162,112 personas >= 5 años.')
save_fig(fig, 'fig08_bivariado_uso_internet_por_educacion_persona')
plt.show()

## 8.5 Dominio Territorial: Brecha Urbana vs. Rural

In [ ]:
tab_zona = df_h.groupby('zona').agg(
 n_hogares=('id_hogar', 'count'),
 pct_internet=('acceso_internet_hogar', lambda x: x.mean() * 100),
 pct_computador=('tiene_computador', lambda x: x.mean() * 100),
 pct_internet_fijo=('tiene_internet_fijo', lambda x: x.mean() * 100),
 pct_internet_movil=('tiene_internet_movil', lambda x: x.mean() * 100)
)

tab_zona['brecha_urb_rur_pp'] = tab_zona.loc['Urbana'] - tab_zona.loc['Rural']
print(tab_zona.round(2))

### Gráfico 9: Comparación de Brechas según Dominio Geográfico (Urbano vs. Rural)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
tab_plot = df_h.groupby('zona').agg(
 internet=('acceso_internet_hogar', lambda x: x.mean() * 100),
 computador=('tiene_computador', lambda x: x.mean() * 100),
 fijo=('tiene_internet_fijo', lambda x: x.mean() * 100)
).T
tab_plot.index = ['Acceso a Internet', 'Tenencia de Computador', 'Conexión Fija']

x = np.arange(len(tab_plot))
w = 0.35
b1 = ax.bar(x - w/2, tab_plot['Urbana'], width=w, label='Urbana (Cabecera)', color='#2b5c8f', edgecolor='white')
b2 = ax.bar(x + w/2, tab_plot['Rural'], width=w, label='Rural (Centros poblados y disperso)', color='#41b6c4', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(tab_plot.index, fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 110)
ax.set_title('Indicadores Tecnológicos según Dominio Geográfico (Urbano vs. Rural)', pad=15)
ax.set_ylabel('Porcentaje de hogares (%)')
ax.legend(frameon=True, facecolor='white', loc='upper right')

for b in b1:
 h = b.get_height()
 ax.text(b.get_x() + b.get_width()/2, h + 1.2, f'{h:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#2b5c8f')
for b in b2:
 h = b.get_height()
 ax.text(b.get_x() + b.get_width()/2, h + 1.2, f'{h:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#006d2c')

plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Muestra: 85,387 hogares.')
save_fig(fig, 'fig09_bivariado_brecha_zona_dispositivos')
plt.show()

## 8.6 Análisis Bivariado Estratificado: Interacción Dificultad Económica $\times$ Zona

In [ ]:
strat_tab = df_h.groupby(['zona', 'alta_dificultad_eco']).agg(
 n_hogares=('id_hogar', 'count'),
 pct_computador=('tiene_computador', lambda x: x.mean() * 100),
 pct_internet=('acceso_internet_hogar', lambda x: x.mean() * 100)
)

print(strat_tab.round(2))

gap_rural = strat_tab.loc[('Rural', 0), 'pct_computador'] - strat_tab.loc[('Rural', 1), 'pct_computador']
gap_urbano = strat_tab.loc[('Urbana', 0), 'pct_computador'] - strat_tab.loc[('Urbana', 1), 'pct_computador']
print(f'\nPenalización por dificultad económica en tenencia de computador:')
print(f' En zona Rural : {gap_rural:.2f} puntos porcentuales')
print(f' En zona Urbana: {gap_urbano:.2f} puntos porcentuales')